In [ ]:
pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 15.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tsfresh 0.21.0 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
opencv-python-h

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Serialization
import json
import pickle

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.pipeline import Pipeline
import warnings
import time
from gensim.models import KeyedVectors
from gensim.utils import simple_preprocess

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Đọc dữ liệu từ file dataset cũ
df = pd.read_csv("/content/drive/MyDrive/SIC/Dataset/Data_Completed.csv")
# Drop các cột có dữ liệu NaN
df = df.dropna(subset=['tokenized_text'])
# Tạo DataFrame mới
df_new = df[['tokenized_text', 'label']].copy()
df_new.rename(columns={'tokenized_text': 'segment_text'}, inplace=True)
# Lưu CSV mới
df_new.to_csv("/content/drive/MyDrive/SIC/Dataset/data_for_ml.csv", index=False, encoding='utf-8-sig')

In [ ]:
# Encode label
le = LabelEncoder()
df['label_int'] = le.fit_transform(df['label'])
# label → int
label_to_int = dict(zip(le.classes_, le.transform(le.classes_)))

# int → label
int_to_label = {v: k for k, v in label_to_int.items()}

# Ép kiểu từ int32 hoặc int64 do LabelEncoder trả về thành int để lưu trong json
label_to_int = {k: int(v) for k, v in label_to_int.items()}
int_to_label = {int(k): v for k, v in int_to_label.items()}

# Gộp thành dict để lưu JSON
mapping_data = {
    "label_to_int": label_to_int,
    "int_to_label": int_to_label
}

In [ ]:
# Lưu mapping vào JSON
with open("/content/drive/MyDrive/SIC/Mapping/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(mapping_data, f, ensure_ascii=False, indent=4)

print("Đã lưu mapping vào label_mapping.json")

Đã lưu mapping vào label_mapping.json


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/SIC/Dataset/data_for_ml.csv", encoding='utf-8-sig')
df = df[['segment_text', 'label']]

In [ ]:
# Load mapping từ JSON
with open("/content/drive/MyDrive/SIC/Mapping/label_mapping.json", "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

In [ ]:
df

,segment_text,label
0,shop phục_vụ rất kém,neg
1,"tôi về vất đi rồi , chỉ được quảng_cáo hay thô...",neg
2,chất_lượng sản_phẩm rất kém đóng_gói sản_phẩm ...,neg
3,bề ngang áo chật,neg
4,và hẳn dây thì tai mèo,neg
...,...,...
14967,mỗi mềm hơn rất nhju,pos
14968,cực_kì đáng tiền,pos
14969,"rẻ , đẹp_trai",pos
14970,"chất_lượng , màu_sắc khá ổn , giống ảnh . khá ...",pos


In [ ]:
int_to_label= mapping_data["int_to_label"]

# Chuyển label thành số từ label mapping
label_to_int = mapping_data["label_to_int"]

## Tạo cột label_int
df["label_int"] = df["label"].map(label_to_int)

# Tách data và label
X = df["segment_text"]
y = df["label_int"]

In [ ]:
# Lần 1: tách test từ dataset
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=42
)

# Lần 2: tách validation từ phần còn lại
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    stratify=y_temp,
    random_state=42
)

print("Train:", y_train.value_counts(normalize=True))
print("Validation:", y_val.value_counts(normalize=True))
print("Test:", y_test.value_counts(normalize=True))

Train: label_int
0    0.333524
2    0.333238
1    0.333238
Name: proportion, dtype: float64
Validation: label_int
0    0.333333
1    0.333333
2    0.333333
Name: proportion, dtype: float64
Test: label_int
0    0.333482
1    0.333482
2    0.333037
Name: proportion, dtype: float64


In [ ]:
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])

In [ ]:
print("\n" + "="*50)
print("LOADING FASTTEXT MODEL")
print("="*50)

# Load FastText model
fasttext_path = "/content/drive/MyDrive/SIC/Fasttext/cc.vi.300.vec"
print(f"Loading FastText model from: {fasttext_path}")

# Load the .vec file
fasttext_model = KeyedVectors.load_word2vec_format(fasttext_path, binary=False)
print(f"FastText model loaded successfully!")
print(f"Vocabulary size: {len(fasttext_model.key_to_index)}")
print(f"Vector dimension: {fasttext_model.vector_size}")


LOADING FASTTEXT MODEL
Loading FastText model from: /content/drive/MyDrive/SIC/Fasttext/cc.vi.300.vec
FastText model loaded successfully!
Vocabulary size: 2000000
Vector dimension: 300


In [ ]:
#FASTTEXT FEATURE EXTRACTION
def text_to_fasttext_vector(text, model, vector_size=300):
    """
    Chuyển đổi text thành vector FastText bằng cách lấy trung bình của các word vectors
    """
    # Tokenize text (vì text đã được tokenized, ta chỉ cần split)
    words = text.split()

    # Lấy vectors của các từ có trong vocabulary
    vectors = []
    for word in words:
        if word in model.key_to_index:
            vectors.append(model[word])

    # Nếu không có từ nào trong vocabulary, trả về vector zero
    if not vectors:
        return np.zeros(vector_size)

    return np.mean(vectors, axis=0)

def texts_to_fasttext_vectors(texts, model, vector_size=300):

    vectors = []
    for i, text in enumerate(texts):
        if i % 1000 == 0:
            print(f"Processing text {i}/{len(texts)}")
        vector = text_to_fasttext_vector(text, model, vector_size)
        vectors.append(vector)

    return np.array(vectors)

In [ ]:
# Tạo vectors cho tất cả texts
print("\n" + "="*50)
print("CREATING FASTTEXT VECTORS")
print("="*50)

print("Creating vectors for training data...")
X_train_vectors = texts_to_fasttext_vectors(X_train_full, fasttext_model)

print("Creating vectors for test data...")
X_test_vectors = texts_to_fasttext_vectors(X_test, fasttext_model)

print(f"Training vectors shape: {X_train_vectors.shape}")
print(f"Test vectors shape: {X_test_vectors.shape}")


CREATING FASTTEXT VECTORS
Creating vectors for training data...
Processing text 0/12726
Processing text 1000/12726
Processing text 2000/12726
Processing text 3000/12726
Processing text 4000/12726
Processing text 5000/12726
Processing text 6000/12726
Processing text 7000/12726
Processing text 8000/12726
Processing text 9000/12726
Processing text 10000/12726
Processing text 11000/12726
Processing text 12000/12726
Creating vectors for test data...
Processing text 0/2246
Processing text 1000/2246
Processing text 2000/2246
Training vectors shape: (12726, 300)
Test vectors shape: (2246, 300)


In [ ]:
 #Chuẩn hóa features (cần thiết cho SVM và Logistic Regression vì các vector có scale lệch)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_vectors)
X_test_scaled = scaler.transform(X_test_vectors)

In [ ]:
print("\nFeature statistics after scaling:")
print(f"Mean: {X_train_scaled.mean():.4f}")
print(f"Std: {X_train_scaled.std():.4f}")
print(f"Min: {X_train_scaled.min():.4f}")
print(f"Max: {X_train_scaled.max():.4f}")


Feature statistics after scaling:
Mean: 0.0000
Std: 1.0000
Min: -18.4042
Max: 17.3223


In [ ]:
with open("/content/drive/MyDrive/SIC/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [ ]:
models = {}
results = {}

In [ ]:
#SUPPORT VECTOR MACHINE
start_SVM = time.time()
classifier_SVM = SVC(
    kernel='linear', # dùng linear vì khi dùng embedding FastText thường tuyến tính
    C=1.0,
    probability=True,
    random_state=42
)
classifier_SVM.fit(X_train_scaled, y_train_full)
end_SVM = time.time()
time_SVM = end_SVM - start_SVM
print(f"Thời gian huấn luyện SVM (Linear Kernel): {time_SVM:.4f} giây")

svm_pred = classifier_SVM.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred, average='weighted')
print(f"SVM (Linear Kernel) - Test Accuracy: {svm_acc:.4f}")
print(f"SVM (Linear Kernel) - Test F1-Score: {svm_f1:.4f}")
print("\nClassification Report for SVM (Linear Kernel):\n", classification_report(y_test, svm_pred, target_names=[int_to_label[i] for i in sorted(int_to_label.keys())]))

models['SVM'] = classifier_SVM
results['SVM'] = {'accuracy': svm_acc, 'f1': svm_f1, 'train_time': time_SVM}

Thời gian huấn luyện SVM (Linear Kernel): 1496.3488 giây
SVM (Linear Kernel) - Test Accuracy: 0.6509
SVM (Linear Kernel) - Test F1-Score: 0.6502

Classification Report for SVM (Linear Kernel):
               precision    recall  f1-score   support

         neg       0.65      0.68      0.67       749
         neu       0.56      0.54      0.55       749
         pos       0.74      0.73      0.73       748

    accuracy                           0.65      2246
   macro avg       0.65      0.65      0.65      2246
weighted avg       0.65      0.65      0.65      2246



In [ ]:
# LOGISTIC REGRESSION
start_LR = time.time()
classifier_LR = LogisticRegression(
    C=1.0,
    penalty='l2',
    solver='saga',   # Tốt cho dữ liệu lớn và hỗ trợ multi-class
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)
classifier_LR.fit(X_train_scaled, y_train_full)
end_LR = time.time()
time_LR = end_LR - start_LR
print(f"Thời gian huấn luyện Logistic Regression: {time_LR:.4f} giây")

# Đánh giá
lr_pred = classifier_LR.predict(X_test_scaled)
lr_acc = accuracy_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred, average='weighted')
print(f"Logistic Regression - Test Accuracy: {lr_acc:.4f}")
print(f"Logistic Regression - Test F1-Score: {lr_f1:.4f}")
print("\nClassification Report for Logistic Regression:\n", classification_report(y_test, lr_pred, target_names=[int_to_label[i] for i in sorted(int_to_label.keys())]))

models['LogisticRegression'] = classifier_LR
results['LogisticRegression'] = {'accuracy': lr_acc, 'f1': lr_f1, 'train_time': time_LR}

Thời gian huấn luyện Logistic Regression: 131.7291 giây
Logistic Regression - Test Accuracy: 0.6465
Logistic Regression - Test F1-Score: 0.6449

Classification Report for Logistic Regression:
               precision    recall  f1-score   support

         neg       0.65      0.66      0.65       749
         neu       0.56      0.53      0.55       749
         pos       0.72      0.75      0.73       748

    accuracy                           0.65      2246
   macro avg       0.64      0.65      0.64      2246
weighted avg       0.64      0.65      0.64      2246



/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:
# RANDOM FOREST
start_RF = time.time()
classifier_RF = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
# Random Forest không yêu cầu dữ liệu phải được scale, nên dùng X_train_vectors
classifier_RF.fit(X_train_vectors, y_train_full)
end_RF = time.time()
time_RF = end_RF - start_RF
print(f"Thời gian huấn luyện Random Forest: {time_RF:.4f} giây")

rf_pred = classifier_RF.predict(X_test_vectors)
rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average='weighted')
print(f"Random Forest - Test Accuracy: {rf_acc:.4f}")
print(f"Random Forest - Test F1-Score: {rf_f1:.4f}")
print("\nClassification Report for Random Forest:\n", classification_report(y_test, rf_pred, target_names=[int_to_label[i] for i in sorted(int_to_label.keys())]))

models['RandomForest'] = classifier_RF
results['RandomForest'] = {'accuracy': rf_acc, 'f1': rf_f1, 'train_time': time_RF}

Thời gian huấn luyện Random Forest: 122.3169 giây
Random Forest - Test Accuracy: 0.6670
Random Forest - Test F1-Score: 0.6677

Classification Report for Random Forest:
               precision    recall  f1-score   support

         neg       0.64      0.69      0.66       749
         neu       0.60      0.59      0.60       749
         pos       0.77      0.72      0.74       748

    accuracy                           0.67      2246
   macro avg       0.67      0.67      0.67      2246
weighted avg       0.67      0.67      0.67      2246



In [ ]:
for model_name, res in results.items():
    print(f"\n{model_name}")
    print(f"Thời gian huấn luyện: {res['train_time']:.4f} giây")
    print(f"Test Accuracy: {res['accuracy']:.4f}")
    print(f"F1-Score (Weighted): {res['f1']:.4f}")


SVM
Thời gian huấn luyện: 1496.3488 giây
Test Accuracy: 0.6509
F1-Score (Weighted): 0.6502

LogisticRegression
Thời gian huấn luyện: 131.7291 giây
Test Accuracy: 0.6465
F1-Score (Weighted): 0.6449

RandomForest
Thời gian huấn luyện: 122.3169 giây
Test Accuracy: 0.6670
F1-Score (Weighted): 0.6677


In [ ]:
model_save_path = "/content/drive/MyDrive/SIC/Model/trained_models.pkl"
results_save_path = "/content/drive/MyDrive/SIC/Model/training_results.json"

In [ ]:
coverage, total_words, covered_words = check_vocab_coverage(X_train_full, fasttext_model)
print(f"\nVocabulary Coverage in Training Data: {coverage:.2%} ({covered_words}/{total_words} words covered)")
unc_words = uncovered_words_list(X_train_full, fasttext_model)
print(f"Top 20 uncovered words: {list(unc_words)[:20]}")


Vocabulary Coverage in Training Data: 84.30% (105657/125339 words covered)
Top 20 uncovered words: ['truy_xuất', 'y_như_vậy', 'gia_hạn', 'phù_hợp', '58kí', 'đại_học', 'dùng_dằng', '30112017', 'trái_lại', 'nhu_vậy', '1sai', 'trở_lên', 'rácậu', 'tổng_đơn', 'sạch_tinh', 'công_an', 'yếu_xìu', 'cổ_tay_áo', 'vả_lại', 'trông_đợi']


In [ ]:
def predict_comment_random_forest(comment, model, fasttext_model, int_to_label):

    vector = text_to_fasttext_vector(comment, fasttext_model).reshape(1, -1)

    pred_label = model.predict(vector)[0]

    proba = model.predict_proba(vector)[0]

    print(f"\nTỉ lệ vote từng class của RandomForest:")
    for idx, prob in enumerate(proba):
        label = int_to_label[str(idx)]
        print(f"{label}: {prob:.4f}")

    print(f"\nPredicted sentiment: {int_to_label[str(pred_label)]}")
    print(f"Max confidence: {max(proba):.4f}")



def predict_comment_scaled_model(comment, model, fasttext_model, scaler, model_name, int_to_label):
    vector = text_to_fasttext_vector(comment, fasttext_model).reshape(1, -1)
    vector_scaled = scaler.transform(vector)
    pred_label = model.predict(vector_scaled)[0]

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(vector_scaled)[0]
        print(f"\nTỉ lệ xác suất từng class của {model_name}:")
        for idx, prob in enumerate(proba):
            label = int_to_label[str(idx)]
            print(f"{label}: {prob:.4f}")
        confidence = max(proba)
    else:
        print(f"\nModel {model_name} không hỗ trợ predict_proba.")
        confidence = None

    print(f"\nPredicted sentiment: {int_to_label[str(pred_label)]}")
    if confidence is not None:
        print(f"Max confidence: {confidence:.4f}")

In [ ]:
comment = "Sản phẩm hơi tệ, nhưng cũng được"

predict_comment_random_forest(comment, models['RandomForest'], fasttext_model, int_to_label)

predict_comment_scaled_model(comment, models['SVM'], fasttext_model, scaler, "SVM", int_to_label)

predict_comment_scaled_model(comment, models['LogisticRegression'], fasttext_model, scaler, "Logistic Regression", int_to_label)



Tỉ lệ vote từng class của RandomForest:
neg: 0.2990
neu: 0.4035
pos: 0.2975

Predicted sentiment: neu
Max confidence: 0.4035

Tỉ lệ xác suất từng class của SVM:
neg: 0.2604
neu: 0.4516
pos: 0.2880

Predicted sentiment: neu
Max confidence: 0.4516

Tỉ lệ xác suất từng class của Logistic Regression:
neg: 0.1542
neu: 0.4541
pos: 0.3917

Predicted sentiment: neu
Max confidence: 0.4541
